# 🕵️ Your First Anonymization

Detect sensitive entities and replace them with LLM-generated substitutes --
the simplest end-to-end example of Anonymizer.

#### 📚 What you'll learn

- Load a CSV dataset and configure Anonymizer in a few lines
- Preview anonymized results on a small sample before committing to a full run
- Inspect entity detection and replacement with `display_record()`
- Process the full dataset with `run()`

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
  - The default `build.nvidia.com` (NVIDIA Build) setup is a convenient way to try Anonymizer and iterate on previews. Use of NVIDIA Build is subject to NVIDIA Build's own terms of service and privacy practices, which are separate from and independent of the NeMo Framework library. NVIDIA Build is intended for evaluation and testing purposes only and may not be used in production environments. Do not upload any confidential information or personal data when using NVIDIA Build. Your use of NVIDIA Build is logged for security purposes and to improve NVIDIA products and services.
  - Request and token rate limits on `build.nvidia.com` vary by account and model access, and lower-volume development access can be slow for full-dataset runs. Start with `preview()` on a small sample, then move to your own endpoint for production data and usage.
- Import the core Anonymizer classes: `Anonymizer`, `AnonymizerConfig`, `AnonymizerInput`, and `Substitute`.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.
- `configure_logging(LoggingConfig.default())` keeps logs at INFO. Switch to `LoggingConfig.debug()` when troubleshooting.

In [2]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [ ]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, LoggingConfig, Substitute, configure_logging

configure_logging(LoggingConfig.default())

In [4]:
anonymizer = Anonymizer()

[13:13:46] [INFO] 🔧 Anonymizer initialized with 3 model configs


[13:13:46] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[13:13:46] [INFO]   |-- ✅ validator: gpt-oss-120b


[13:13:46] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Load data and configure

- `AnonymizerInput` points to your CSV and names the text column. `data_summary`
  gives the LLM context about the kind of text it will process.
- Records up to 2,000 tokens each work with the default model configs.
- `AnonymizerConfig` with `Substitute()` tells Anonymizer to replace detected
  entities with LLM-generated synthetic values for names, cities, dates, etc.

In [5]:
input_data = AnonymizerInput(
    source="https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles of individuals",
)

config = AnonymizerConfig(replace=Substitute())

## 👁️ Preview

- `preview()` runs on a small sample so you can iterate quickly.
- Always preview before processing the full dataset -- it's the fastest way
  to catch prompt or config issues early.

In [6]:
preview = anonymizer.preview(config=config, data=input_data, num_records=3)

[13:13:46] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[13:13:46] [INFO] 🔍 Running entity detection on 3 records


[13:13:46] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[13:14:17] [INFO]   |-- 📋 Detection complete — 80 entities found across 3 records (0 failed) [30.6s]


[13:14:17] [INFO]   |-- labels: first_name=23, state=6, organization_name=6, age=5, occupation=5, city=5, company_name=4, last_name=3, race_ethnicity=3, language=3, political_view=3, education_level=3, field_of_study=2, religious_belief=2, street_address=2, degree=1, university=1, place_name=1, date_of_birth=1, employment_status=1


[13:14:17] [INFO] 🔄 Running Substitute replacement


[13:15:14] [INFO]   |-- 📋 Replacement complete (0 failed) [57.4s]


[13:15:14] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


## 🔍 Inspect

- `display_record()` shows the original text with highlighted entities,
  the replacement map, and the anonymized output -- all in one view.
- The result dataframe has original and substituted text side-by-side.

In [7]:
preview.display_record(0)

Original,Label,Replacement
40,age,45
Aria,first_name,Sofia
Bobby,first_name,Ethan
Christian Democrat,political_view,Libertarian
Colorado,state,Oregon
Colorado Veterinary Clinic,organization_name,Oregon Animal Wellness Center
DVM,degree,Ph.D.
Denver,city,Portland
English,language,Spanish
Jefferson High,organization_name,Lincoln High


In [8]:
preview.display_record(1)

Original,Label,Replacement
37,age,36
Bell,last_name,Kumar
Edison,city,Austin
Elena,first_name,Sofia
English,language,Spanish
Goddard Space Flight Center,organization_name,National Renewable Energy Laboratory
Idilio,first_name,Santiago
Italian,race_ethnicity,Greek
Lina,first_name,Aisha
Marco,first_name,Diego


In [9]:
preview.dataframe

,biography,biography_with_spans,final_entities,biography_replaced
0,"Bobby Watford, a 40‑year‑old Mexican veterinar...",<first_name>Bobby</first_name> <last_name>Watf...,"{'entities': [{'end_position': 5, 'id': 'first...","Ethan Henderson, a 45‑year‑old Vietnamese mari..."
1,Idilio Bell is a 37‑year‑old astronomer living...,<first_name>Idilio</first_name> <last_name>Bel...,"{'entities': [{'end_position': 6, 'id': 'first...",Santiago Kumar is a 36‑year‑old geophysicist l...
2,"Jodi Allison, 36, lives at 204 Bluegrass in Cl...",<first_name>Jodi</first_name> <last_name>Allis...,"{'entities': [{'end_position': 4, 'id': 'first...","Sofia Keller, 42, lives at 587 Maple in Macon,..."


## 🚀 Full run

- `run()` processes the entire dataset with the same config you previewed.
- Access the output via `result.dataframe`.

In [10]:
result = anonymizer.run(config=config, data=input_data)
print(result)

[13:15:14] [INFO] 📂 Loaded 25 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[13:15:14] [INFO] 🔍 Running entity detection on 25 records


[13:15:14] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[13:16:05] [INFO]   |-- 📋 Detection complete — 648 entities found across 25 records (0 failed) [50.3s]


[13:16:05] [INFO]   |-- labels: first_name=152, city=48, occupation=45, company_name=40, education_level=33, race_ethnicity=31, state=30, organization_name=30, last_name=27, age=26, political_view=26, religious_belief=25, street_address=23, university=21, language=21, field_of_study=13, place_name=12, county=11, employment_status=10, date_of_birth=9, date=5, degree=4, school_name=1, landmark=1, journal_name=1, country=1, gender=1, postcode=1


[13:16:05] [INFO] 🔄 Running Substitute replacement


[13:16:35] [INFO]   |-- 📋 Replacement complete (0 failed) [30.5s]


[13:16:35] [INFO] 🎉 Pipeline complete — 25 records processed, 0 total failures


AnonymizerResult(rows=25, columns=4, trace_columns=21, failed_records=0)


In [11]:
result.dataframe.head()

,biography,biography_with_spans,final_entities,biography_replaced
0,"Bobby Watford, a 40‑year‑old Mexican veterinar...",<first_name>Bobby</first_name> <last_name>Watf...,"{'entities': array([{'end_position': 5, 'id': ...","Ethan Hernandez, a 52‑year‑old Filipino zoolog..."
1,Idilio Bell is a 37‑year‑old astronomer living...,<first_name>Idilio</first_name> <last_name>Bel...,"{'entities': array([{'end_position': 6, 'id': ...",Rafael Khan is a 42‑year‑old planetary geologi...
2,"Jodi Allison, 36, lives at 204 Bluegrass in Cl...",<first_name>Jodi</first_name> <last_name>Allis...,"{'entities': array([{'end_position': 4, 'id': ...","Leah Harper, 42, lives at 204 Willow in Eugene..."
3,James Mills is a 69‑year‑old paramedic who liv...,<first_name>James</first_name> <last_name>Mill...,"{'entities': array([{'end_position': 5, 'id': ...",Ethan Harper is a 71‑year‑old firefighter who ...
4,Nancy Burton is a 21‑year‑old cashier who live...,<first_name>Nancy</first_name> <last_name>Burt...,"{'entities': array([{'end_position': 5, 'id': ...",Leah Hawkins is a 27‑year‑old stock clerk who ...


## ⏭️ Next steps

- **[🔍 Inspecting Detected Entities](../02_inspecting_detected_entities/)** --
  dig into what the detection pipeline found and debug quality.
- **[🎯 Choosing a Replacement Strategy](../03_choosing_a_replacement_strategy/)** --
  compare Redact, Annotate, Hash, and Substitute side-by-side.
- **[✏️ Rewriting Biographies](../04_rewriting_biographies/)** --
  generate privacy-safe paraphrases instead of token-level replacements.